<a href="https://colab.research.google.com/github/jacobhroutzong/Reinforcement-Learning/blob/main/Sentinyl_Triage_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# # Sentinyl Triage Agent (Standard Stack 2026)
# **Stack:** LangGraph (Orchestration), Pydantic (Schema), Claude (Brain), Weaviate (Memory)

# %%
import operator
from typing import Annotated, List, TypedDict, Union
from langgraph.graph import StateGraph, END

# --- 1. THE SCHEMA (The "Shape" of your data) ---
# In 2026, we don't ask for "text back". We ask for Objects.
from pydantic import BaseModel, Field

class ThreatAssessment(BaseModel):
    risk_score: int = Field(description="0-100 score of the threat")
    classification: str = Field(description="'Critical', 'Warning', or 'False Positive'")
    reasoning: str = Field(description="Why you made this decision")

# --- 2. THE STATE (The "Memory" of the Agent) ---
# This dictionary lives in the graph and passes data between nodes.
class AgentState(TypedDict):
    alert_id: str
    raw_logs: str
    context_docs: Annotated[List[str], operator.add] # 'add' means new docs are appended, not overwritten
    assessment: dict # Stores the final Pydantic output

# --- 3. THE NODES (The "Workers" in your factory) ---

def retrieve_context(state: AgentState):
    """
    Simulates querying Weaviate/LlamaIndex for similar past threats.
    In a real app, this would be: index.as_retriever().retrieve(state['raw_logs'])
    """
    print(f"🔎 querying Weaviate for alert: {state['alert_id']}...")

    # Mock retrieved data
    retrieved_info = [
        "KnowledgeBase: User 'admin' usually logs in from IP 192.168.1.5",
        "History: Similar pattern seen 2 weeks ago (False Positive)"
    ]
    return {"context_docs": retrieved_info}

def analyze_threat(state: AgentState):
    """
    The 'Brain' (Claude 3.5 Sonnet).
    It looks at the logs + the retrieved context to make a decision.
    """
    print("🧠 Claude is thinking...")

    # Mocking the LLM interaction
    # In reality: response = llm.with_structured_output(ThreatAssessment).invoke(...)

    # Let's pretend Claude decided this is a False Positive based on history
    decision = ThreatAssessment(
        risk_score=15,
        classification="False Positive",
        reasoning="User behavior matches known safe pattern from 2 weeks ago."
    )

    return {"assessment": decision.model_dump()}

def human_review(state: AgentState):
    """Optional: If confidence is low, route here."""
    print("Drafting slack message to human analyst...")
    return {}

# --- 4. THE EDGES (The Logic Flow) ---

def router(state: AgentState):
    # If risk is high, loop or escalate. If low, close it.
    score = state["assessment"]["risk_score"]
    if score > 80:
        return "human_review"
    else:
        return "end"

# --- 5. BUILD THE GRAPH ---

workflow = StateGraph(AgentState)

# Add Nodes
workflow.add_node("retrieve_knowledge", retrieve_context)
workflow.add_node("analyst_brain", analyze_threat)
workflow.add_node("human_review", human_review)

# Add Edges (The Flow)
workflow.set_entry_point("retrieve_knowledge")
workflow.add_edge("retrieve_knowledge", "analyst_brain")

# Conditional Edge (The Decision)
workflow.add_conditional_edges(
    "analyst_brain",
    router,
    {
        "human_review": "human_review",
        "end": END
    }
)

# Compile (Freezes the graph structure)
app = workflow.compile()

# %% [markdown]
# ### Run the Agent
# We pass in a raw log, and the agent automatically fetches context and makes a decision.

# %%
inputs = {
    "alert_id": "ALERT-9921",
    "raw_logs": "SSH login attempt for user 'admin' from IP 192.168.1.5",
    "context_docs": []
}

# Run the graph
result = app.invoke(inputs)

print(f"\n✅ Final Decision: {result['assessment']['classification']}")
print(f"📝 Reasoning: {result['assessment']['reasoning']}")

🔎 querying Weaviate for alert: ALERT-9921...
🧠 Claude is thinking...

✅ Final Decision: False Positive
📝 Reasoning: User behavior matches known safe pattern from 2 weeks ago.
